In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import re
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer

d:\Portfolio\Machine_Learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [3]:
csv_path = os.path.join("..", "data", "preprocessed", "final_dataset.csv")
df = pd.read_csv(csv_path)

print(df.head())
print(df.columns)

                                                text primary_emotion  \
0                            i didnt feel humiliated             sad   
1  i can go from feeling so hopeless to so damned...             sad   
2   im grabbing a minute to post i feel greedy wrong           angry   
3  i am ever feeling nostalgic about the fireplac...           happy   
4                               i am feeling grouchy           angry   

  sub_emotions  
0      ['sad']  
1      ['sad']  
2    ['angry']  
3     ['love']  
4    ['angry']  
Index(['text', 'primary_emotion', 'sub_emotions'], dtype='str')


In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = text.strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)

print(df[['text', 'clean_text']].head())

                                                text  \
0                            i didnt feel humiliated   
1  i can go from feeling so hopeless to so damned...   
2   im grabbing a minute to post i feel greedy wrong   
3  i am ever feeling nostalgic about the fireplac...   
4                               i am feeling grouchy   

                                          clean_text  
0                            i didnt feel humiliated  
1  i can go from feeling so hopeless to so damned...  
2   im grabbing a minute to post i feel greedy wrong  
3  i am ever feeling nostalgic about the fireplac...  
4                               i am feeling grouchy  


In [5]:
label_encoder = LabelEncoder()

df['label'] = label_encoder.fit_transform(df['primary_emotion'])

print(label_encoder.classes_)  # important for later
print(df[['primary_emotion', 'label']].head())

['angry' 'fear' 'happy' 'neutral' 'sad' 'surprise']
  primary_emotion  label
0             sad      4
1             sad      4
2           angry      0
3           happy      2
4           angry      0


In [6]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df['label'],
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))

Train size: 42954
Validation size: 4773


In [7]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [8]:
def tokenize(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

train_encodings = tokenize(train_df['clean_text'])
val_encodings = tokenize(val_df['clean_text'])

In [9]:
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [10]:
train_dataset = EmotionDataset(
    train_encodings,
    train_df['label'].tolist()
)

val_dataset = EmotionDataset(
    val_encodings,
    val_df['label'].tolist()
)

In [11]:
sample = train_dataset[0]

print(sample.keys())
print(sample['input_ids'][:10])
print(sample['attention_mask'][:10])
print("Label:", sample['labels'])

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
tensor([  101,  2179,  1996, 14337,  1998, 28543,  3424,  3567, 20348,  2121])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
Label: tensor(0)


In [12]:
from transformers import BertForSequenceClassification

num_labels = len(label_encoder.classes_)

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2491.54it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/primary_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="../logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [14]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [17]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [18]:
trainer.train()

d:\Portfolio\Machine_Learning\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.evaluate()